In [1]:
from pathlib import Path
import pandas as pd
 
from src import (
    TransferEvaluator,
    EvaluationResult,
    CompositeDistanceRanker,
    SingleFeatureRanker,
    LightGBMRanker,
    MLPRanker
)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [2]:
DATA_PATH = Path('data/sib200.csv')
RESULTS_DIR = Path('results')
PERFORMANCE_COL = 'f1_score'
FEATURE_COLS = ['new_typ', 'new_geo', 'new_gen', 'script']
BASELINE = 'composite_uniform'

In [5]:
rankers = {}
rankers['composite_uniform'] = CompositeDistanceRanker()
for i, feat in enumerate(FEATURE_COLS):
    rankers[f'only_{feat}'] = SingleFeatureRanker(feature_idx=i)
rankers['lgbm'] = LightGBMRanker(n_estimators=100, num_leaves=16, learning_rate=0.1)
rankers['mlp'] = MLPRanker(epochs=50, hidden_dims=tuple([8]), patience=20)

In [6]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows | "
      f"{df['task_lang'].nunique()} targets | "
      f"{df['transfer_lang'].nunique()} sources | "
      f"features: {FEATURE_COLS}")

evaluator = TransferEvaluator(performance_col=PERFORMANCE_COL, verbose=False)

results = {}
for name, ranker in rankers.items():
    print(f"evaluating {name}...")
    results[name] = evaluator.evaluate(ranker, df, FEATURE_COLS)

Loaded 34200 rows | 190 targets | 180 sources | features: ['new_typ', 'new_geo', 'new_gen', 'script']
evaluating composite_uniform...


100%|██████████| 190/190 [00:07<00:00, 24.53it/s]


evaluating only_new_typ...


100%|██████████| 190/190 [00:07<00:00, 26.66it/s]


evaluating only_new_geo...


...

100%|██████████| 190/190 [00:07<00:00, 25.90it/s]


evaluating lgbm...


100%|██████████| 190/190 [02:30<00:00,  1.26it/s]


evaluating mlp...


100%|██████████| 190/190 [11:25<00:00,  3.61s/it]


In [7]:
rows = []
base_res = results[BASELINE]
for name, res in results.items():
    row = {
        'method': name,
        'ndcg@3': res.mean_ndcg,
        'perf_loss': res.mean_performance_loss,
        'top1_acc': res.mean_top_1_accuracy,
        'top3_acc': res.mean_top_3_accuracy,
        'p_vs_baseline': (
            None if name == BASELINE
            else TransferEvaluator.compare(res, base_res)['ndcg_p_value']
        ),
    }
    rows.append(row)
summary_df = pd.DataFrame(rows)

In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'per_fold').mkdir(exist_ok=True)
summary_df.to_csv(RESULTS_DIR / 'summary.csv', index=False)
for name, res in results.items():
    res.per_fold.to_csv(RESULTS_DIR / 'per_fold' / f'{name}.csv', index=False)

print(f"\nResults saved to {RESULTS_DIR.resolve()}")
print('\n=== Summary ===')
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))


Results saved to /Users/yorky/langrankplus/results

=== Summary ===
           method  ndcg@3  perf_loss  top1_acc  top3_acc  p_vs_baseline
composite_uniform  19.405     22.488    13.684    17.368            NaN
     only_new_typ  17.365     22.986    11.579    14.211          0.031
     only_new_geo  18.065     24.013    13.684    16.842          0.085
     only_new_gen  19.160     22.355    13.684    17.368          0.329
      only_script   7.999     23.558     3.158     5.789          0.000
             lgbm  19.610     22.783    13.684    16.316          0.813
              mlp   1.295     54.234     0.000     0.526          0.000
